In [231]:
import json
import os
import dotenv

import numpy as np
import pandas as pd

from sklearn.cluster import HDBSCAN

import sqlalchemy as sa
from sqlalchemy import select, create_engine

from models import Wards, Matches

dotenv.load_dotenv()

True

In [232]:
db = create_engine(os.environ['DB_URL'])

In [233]:
df = pd.DataFrame(pd.read_sql(
    sql=select(Wards, Matches).join(Matches).where(sa.and_(Wards.is_obs, Matches.league_id == 15728)).order_by(Matches.id.desc()),
    con=db
))

In [234]:
df['duration'] = df[['duration', 'duration_1', 'time_placed']].apply((lambda row: min(row['duration'], row['duration_1'] - row['time_placed'])), axis=1)

In [235]:
hdbscan = HDBSCAN(
    cluster_selection_epsilon=70,
    min_samples=5,
    min_cluster_size=5,
    cluster_selection_method='eom',
    n_jobs=-1,
)
hdbscan.fit(df[['x_pos', 'y_pos']])

df['cluster_id'] = hdbscan.labels_

In [236]:
df.groupby('cluster_id').size().sort_values(ascending=False)

cluster_id
-1      628
 66     236
 36     211
 4      151
 3      150
       ... 
 93       5
 9        5
 160      5
 84       5
 73       5
Length: 176, dtype: int64

In [237]:
df['advantage'] = df[['is_radiant', 'radiant_networth', 'dire_networth']].apply(lambda row: (row['radiant_networth'] - row['dire_networth']) if row['is_radiant'] else (row['dire_networth'] - row['radiant_networth']), axis=1)
df

,id,match_id,is_obs,is_radiant,is_destroyed,is_post_game,player_placed_id,player_destroyed_id,time_placed,duration,...,radiant_networth,dire_networth,id_1,league_id,duration_1,region,date,game_version,cluster_id,advantage
0,837950,7406531302,True,True,False,False,16497807.0,NaN,1181,360,...,6275,8248,7406531302,15728,2525,1,1698621223,53,66,-1973
1,837989,7406531302,True,True,True,False,91730177.0,256156323.0,2292,75,...,15038,22706,7406531302,15728,2525,1,1698621223,53,55,-7668
2,837987,7406531302,True,True,True,False,91730177.0,256156323.0,2095,180,...,14512,19653,7406531302,15728,2525,1,1698621223,53,97,-5141
3,837981,7406531302,True,True,True,False,16497807.0,302214028.0,2118,31,...,14551,20132,7406531302,15728,2525,1,1698621223,53,110,-5581
4,837973,7406531302,True,True,True,False,91730177.0,256156323.0,1907,100,...,12444,16485,7406531302,15728,2525,1,1698621223,53,98,-4041
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5370,827118,7378530387,True,True,True,False,10366616.0,10366616.0,-57,353,...,700,600,7378530387,15728,1405,1,1697130044,53,36,100
5371,827122,7378530387,True,True,True,False,94786276.0,346412363.0,352,15,...,2615,1067,7378530387,15728,1405,1,1697130044,53,152,1548
5372,827124,7378530387,True,False,True,False,238239590.0,94054712.0,447,27,...,3334,1349,7378530387,15728,1405,1,1697130044,53,141,-1985
5373,827130,7378530387,True,True,True,False,94786276.0,238239590.0,451,191,...,3340,1355,7378530387,15728,1405,1,1697130044,53,-1,1985


In [305]:
grouped = df.groupby(['cluster_id', 'is_radiant']).agg({
        'x_pos': lambda x: round(x.mean()),
        'y_pos': lambda x: round(x.mean()),
        'z_pos': lambda x: x.mode().iloc[0], #- 16384,
        'duration': lambda x: round(x.mean()),
        'time_placed': lambda x: round(x.mean()),
        'advantage': lambda x: round(x.mean()),
        'is_destroyed': 'sum'
    })
grouped['amount'] = df.groupby(['cluster_id', 'is_radiant']).size()
grouped

x_pos  y_pos  z_pos  duration  time_placed  advantage  \
cluster_id is_radiant                                                          
-1         False       15501  16097  16512       276         1226        728   
           True        16870  16639  16512       263         1249       -451   
 0         False       14841  23284  16784       285         1697       -801   
           True        14827  23290  16784       193         1522       1528   
 1         False        8450  18193  16920       289         1661         77   
...                      ...    ...    ...       ...          ...        ...   
 172       True        18719  13503  16512       360          419       -464   
 173       False       17566  12902  16640       201         1051       1962   
           True        17509  12928  16640       270          931      -1958   
 174       False       17788  12831  16640       273          954       -298   
           True        17806  12824  16640       309          934       1148   

                       is_destroyed  amount  
cluster_id is_radiant                        
-1         False                112     326  
           True                 114     302  
 0         False                 15      44  
           True                  30      48  
 1         False                  8      27  
...                             ...     ...  
 172       True                   0       7  
 173       False                  2       3  
           True                   2       6  
 174       False                  4      10  
           True                   1       6  

[333 rows x 8 columns]

In [239]:
players = df[df['cluster_id'] != -1].groupby(['cluster_id', 'player_placed_id']).agg({'duration': lambda x: round(x.mean())})
players['amount'] = df.groupby(['cluster_id', 'player_placed_id']).size()
players

duration  amount
cluster_id player_placed_id                  
0          10366616.0              36       1
           77490514.0             298       2
           81306398.0             202       4
           81475303.0             266      11
           85696663.0             360       2
...                               ...     ...
174        161839895.0            360       1
           193815691.0            193       1
           256156323.0             98       1
           346412363.0            360       1
           431770905.0            150       1

[2493 rows x 2 columns]

In [240]:
players_destroyed = df[df['cluster_id'] != -1].groupby(['cluster_id', 'player_destroyed_id']).agg({'duration': lambda x: round(x.mean())})
players_destroyed['amount'] = df.groupby(['cluster_id', 'player_destroyed_id']).size()
players_destroyed

duration  amount
cluster_id player_destroyed_id                  
0          10366616.0                 42       1
           16497807.0                 37       1
           25907144.0                105       2
           77490514.0                106       3
           81475303.0                108       2
...                                  ...     ...
174        54580962.0                 98       1
           87063175.0                126       1
           101695162.0               150       1
           161839895.0                56       1
           317880638.0               193       1

[1301 rows x 2 columns]

In [ ]:
for i in range(len(set(hdbscan.labels_)) - 1):
    current = grouped.loc[i]

    d = {
        'cluster_id': i,
        'x_pos': round(current['x_pos'].mean()),
        'y_pos': round(current['y_pos'].mean()),
        'z_pos': round(current['z_pos'].mean()),
        'radiant': {
            'amount': current.loc[True]['amount'],
            'destroyed': current.loc[True]['is_destroyed'],
            'advantage': current.loc[True]['advantage'],
            'duration': current.loc[True]['duration'],
            'time_placed': current.loc[True]['time_placed']
        } if True in current.index else None,
        'dire': {
            'amount': current.loc[False]['amount'],
            'destroyed': current.loc[False]['is_destroyed'],
            'advantage': current.loc[False]['advantage'],
            'duration': current.loc[False]['duration'],
            'time_placed': current.loc[False]['time_placed']
        } if False in current.index else None,
        'all': {
            'amount': current['amount'].sum(),
            'destroyed': current['is_destroyed'].sum(),
            'advantage': None,
            'duration': round(current['duration'].mean()),
            'time_placed': round(current['time_placed'].mean())
        }
    }

    print(json.dumps(d, indent=4, default=int))

In [ ]:
average_df = grouped.reset_index('is_radiant')
average_radiant = average_df[average_df['is_radiant'] == True]
average_dire = average_df[average_df['is_radiant'] == False]
average = {
    'cluster_id': -1,
    'radiant': {
        'advantage': round(average_radiant['advantage'].mean()),
        'duration': round(average_radiant['duration'].mean()),
    },
    'dire': {
        'advantage': round(average_dire['advantage'].mean()),
        'duration': round(average_dire['duration'].mean()),
    },
    'all': {
        'advantage': None,
        'duration': round(average_df['duration'].mean()),
    }
}
average